# EFM Cosmic Chronology Validation

This notebook performs the Lagrangian stability tracking required to validate the *Cosmic Chronology* paper.
It reads the checkpoint files from the `StructureFormation V12` run (N=1024) and calculates the **Radius of Gyration ($R_g$)** for the most massive emergent solitons across cosmic time.

**Objective:**
Identify sharp drops in $R_g$ corresponding to the Great Recrystallization (Step 240,000) and the Structural Snap (Step 262,000).

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
import gc
from tqdm.notebook import tqdm

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not in Google Colab environment.")

In [ ]:
# Configuration
data_path = '/content/drive/My Drive/EFM_Simulations/data/FirstPrinciples_Dynamic_N1024_v12_StructureFormation/'
checkpoint_pattern = os.path.join(data_path, 'CHECKPOINT_step_*.npz')
checkpoints = glob.glob(checkpoint_pattern)

def get_step(filename):
    try:
        return int(os.path.basename(filename).split('_')[2])
    except:
        return -1

checkpoints = sorted([c for c in checkpoints if get_step(c) != -1], key=get_step)
print(f"Found {len(checkpoints)} checkpoint files.")

In [ ]:
# EFM Parameter (Density Coupling)
k_density_coupling = 0.01

history_rg = {} # Key: step, Value: list of top 5 R_g

# Perform Lagrangian Stability Tracking across all steps
for cp_file in tqdm(checkpoints, desc="Processing Checkpoints"):
    step = get_step(cp_file)
    try:
        with np.load(cp_file, allow_pickle=True) as data:
            phi = data['phi_cpu'].astype(np.float32)
    except Exception as e:
        print(f"Error loading {cp_file}: {e}")
        continue
    
    # Calculate field density
    rho = k_density_coupling * (phi ** 2)
    
    # Statistical threshold for identifying hard matter / solitons
    threshold = np.percentile(rho, 99.99)
    mask = rho > threshold
    
    labeled_array, num_features = label(mask)
    
    if num_features == 0:
        history_rg[step] = []
        del phi, rho, mask, labeled_array
        gc.collect()
        continue
        
    labels, counts = np.unique(labeled_array, return_counts=True)
    
    # Exclude background (label 0)
    valid_idx = labels > 0
    labels = labels[valid_idx]
    counts = counts[valid_idx]
    
    # Identify the Top 5 most massive objects (solitons)
    top_indices = np.argsort(counts)[::-1][:5]
    top_labels = labels[top_indices]
    
    step_rg = []
    
    for lbl in top_labels:
        soliton_mask = (labeled_array == lbl)
        coords = np.argwhere(soliton_mask)
        if len(coords) == 0:
            continue
            
        # Weighted center of mass
        masses = rho[soliton_mask]
        total_mass = masses.sum()
        
        com = np.average(coords, axis=0, weights=masses)
        
        # Calculate Radius of Gyration: R_g = sqrt( sum( m_i * (r_i - r_cm)^2 ) / M )
        diffs = coords - com
        sq_dists = np.sum(diffs**2, axis=1)
        rg = np.sqrt(np.sum(masses * sq_dists) / total_mass)
        step_rg.append(rg)
        
    history_rg[step] = step_rg
    
    del phi, rho, mask, labeled_array
    gc.collect()

In [ ]:
# Plotting the resulting Lagrangian Stability Timeline
steps = sorted(history_rg.keys())
rg_alpha = []

for s in steps:
    rgs = history_rg[s]
    if len(rgs) > 0:
        rg_alpha.append(rgs[0]) # Tracking only the Alpha Core for simplicity in main plot
    else:
        rg_alpha.append(np.nan)

plt.figure(figsize=(14, 8))
plt.plot(steps, rg_alpha, label='Alpha Core (Most Massive Soliton)', color='red', linewidth=2.5)

# Annotate critical EFM Cosmic Chronology events
plt.axvline(x=130000, color='orange', linestyle='--', linewidth=2, label='Firstborn Stabilization (~130k)')
plt.axvline(x=240000, color='blue', linestyle='--', linewidth=2, label='Great Recrystallization (~240k | 1.4 Ga)')
plt.axvline(x=262000, color='purple', linestyle='--', linewidth=2, label='Structural Snap (~262k | 258 Ma)')

plt.title('Lagrangian Stability Tracking: Search for the Structural Snap', fontsize=18)
plt.xlabel('Simulation Step', fontsize=14)
plt.ylabel('Radius of Gyration ($R_g$)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()

output_plot = os.path.join(data_path, 'Cosmic_Chronology_Analysis.png')
plt.savefig(output_plot, dpi=200)
plt.show()

print(f"\nAnalysis complete. Plot saved to Drive at: {output_plot}")
print("Review the plot for the signature sudden drops in R_g correlating with the phase transitions.")